# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandas numpy matplotlib seaborn textblob statsmodels emoji wordcloud Faker
    print('Packages installed.')

# Capstone Project: End-to-End Social Media Analysis

This notebook represents a comprehensive workflow, taking you from raw data generation to advanced causal inference. It integrates skills learned throughout the previous notebooks.

**Scenario:**
You are a social scientist analyzing the impact of a new digital well-being policy introduced on **July 1st, 2023**. 
*   **Treatment Group**: Users in the "South" region (where the policy was enforced).
*   **Control Group**: Users in the "North" region (no policy).
*   **Outcome**: Sentiment of social media posts.

**Steps:**
1.  **Data Simulation**: Generating realistic text and metadata.
2.  **Preprocessing**: Cleaning text and extracting sentiment.
3.  **EDA**: Visualizing trends over time.
4.  **Causal Inference**: Using Difference-in-Differences (DiD) to estimate the policy's effect.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
from textblob import TextBlob
import random
from datetime import datetime, timedelta
import statsmodels.formula.api as smf

# Settings
np.random.seed(42)
fake = Faker()
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Simulation
We create a synthetic dataset of 1,000 users and 5,000 posts spanning 6 months (April - September 2023). The policy starts on July 1st.

In [ ]:
# Parameters
n_users = 500
n_posts = 3000
policy_date = datetime(2023, 7, 1)

# Generate Users
users = []
for _ in range(n_users):
    uid = fake.uuid4()[:8]
    region = random.choice(['North', 'South']) # North = Control, South = Treatment
    users.append({'User_ID': uid, 'Region': region})
df_users = pd.DataFrame(users)

# Vocabulary for sentiment
positive_words = ['happy', 'great', 'love', 'amazing', 'good', 'excited', 'wonderful']
negative_words = ['sad', 'hate', 'terrible', 'bad', 'angry', 'frustrated', 'annoyed']
neutral_words = ['today', 'went', 'saw', 'meeting', 'lunch', 'weather', 'work']

def generate_text(sentiment_bias):
    words = []
    # Add bias words
    if sentiment_bias > 0:
        words.extend(random.choices(positive_words, k=2))
    elif sentiment_bias < 0:
        words.extend(random.choices(negative_words, k=2))
    
    # Fill with neutral/random text
    words.extend(random.choices(neutral_words, k=5))
    random.shuffle(words)
    return " ".join(words)

# Generate Posts
posts = []
start_date = datetime(2023, 4, 1)
end_date = datetime(2023, 9, 30)
days_range = (end_date - start_date).days

for _ in range(n_posts):
    user = random.choice(users)
    region = user['Region']
    
    # Random Date
    post_date = start_date + timedelta(days=random.randint(0, days_range))
    
    # Treatment Effect Logic:
    # After policy_date, 'South' users become slightly more positive (policy works!)
    sentiment_bias = 0
    if region == 'South' and post_date >= policy_date:
        sentiment_bias = 0.5 # Positive shift
    else:
        # Random noise
        sentiment_bias = np.random.uniform(-0.5, 0.5)
        
    text = generate_text(sentiment_bias)
    
    posts.append({
        'User_ID': user['User_ID'],
        'Region': region,
        'Date': post_date,
        'Text': text
    })

df = pd.DataFrame(posts)
df = df.sort_values('Date')
print(f"Generated {len(df)} posts.")
df.head()

## 2. NLP: Sentiment Analysis
We use `TextBlob` to score each post's sentiment (Polarity: -1 to +1).

In [ ]:
def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

df['Sentiment'] = df['Text'].apply(get_sentiment)

# Histogram of Sentiment
sns.histplot(df['Sentiment'], bins=20, kde=True)
plt.title('Distribution of Sentiment Scores')
plt.show()

## 3. Exploratory Analysis
Let's aggregate sentiment by Month and Region to visualize the trend.

In [ ]:
df['Month'] = df['Date'].dt.to_period('M') # Monthly period

# Aggregate
monthly_sentiment = df.groupby(['Month', 'Region'])['Sentiment'].mean().reset_index()

# Convert Month back to timestamp for plotting
monthly_sentiment['Month_Ts'] = monthly_sentiment['Month'].dt.to_timestamp()

# Plot
plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_sentiment, x='Month_Ts', y='Sentiment', hue='Region', marker='o', linewidth=2.5)

# Mark Policy Date
plt.axvline(policy_date, color='red', linestyle='--', label='Policy Start')
plt.title('Average Sentiment by Region over Time')
plt.legend()
plt.show()

## 4. Causal Inference: Difference-in-Differences (DiD)
Visual inspection suggests the South (Treatment) improved relative to the North (Control) after July. Let's quantify this.

**Model:**
$$ Sentiment = \beta_0 + \beta_1(Treatment) + \beta_2(Post) + \beta_3(Treatment \times Post) + \epsilon $$

*   `Treatment`: 1 if Region is South, 0 otherwise.
*   `Post`: 1 if Date >= July 1st, 0 otherwise.
*   **`Treatment x Post`**: The causal effect (DiD estimator).


In [ ]:
# Create Dummy Variables
df['Treatment'] = (df['Region'] == 'South').astype(int)
df['Post'] = (df['Date'] >= policy_date).astype(int)

# Run OLS Regression
model = smf.ols("Sentiment ~ Treatment * Post", data=df).fit()

print(model.summary())

# Extract Effect
did_coeff = model.params['Treatment:Post']
p_value = model.pvalues['Treatment:Post']

print(f"\nEstimated Causal Effect: {did_coeff:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: The policy had a statistically significant effect.")
else:
    print("Result: No significant effect detected.")

## Conclusion
In this capstone, we:
1.  Simulated social media data.
2.  Extracted sentiment using NLP.
3.  Visualized trends.
4.  Applied a rigorous Causal Inference method (DiD) to prove the impact of an intervention.

This workflow demonstrates how to combine computational tools (Python, NLP) with statistical reasoning to answer social science questions.